<a href="https://colab.research.google.com/github/Mizharrrrrhidi1818/EnsembleMethod-SingleModel-Bagging-Stacking/blob/main/EnsembleMethod_SingleModel_Bagging_Stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd, numpy as np
from pandas import DataFrame, Series
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abilashcheruvathur/vehicle-shillhote")

print("Path to dataset files:", path)

100%|██████████| 19.4k/19.4k [00:00<00:00, 23.5MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/abilashcheruvathur/vehicle-shillhote/versions/1


In [ ]:
path

'/root/.cache/kagglehub/datasets/abilashcheruvathur/vehicle-shillhote/versions/1'

In [ ]:
import pandas as pd
df = pd.read_csv(path + "/vehicle.csv")

df.head()

,compactness,circularity,distance_circularity,radius_ratio,pr.axis_aspect_ratio,max.length_aspect_ratio,scatter_ratio,elongatedness,pr.axis_rectangularity,max.length_rectangularity,scaled_variance,scaled_variance.1,scaled_radius_of_gyration,scaled_radius_of_gyration.1,skewness_about,skewness_about.1,skewness_about.2,hollows_ratio,class
0,95,48.0,83.0,178.0,72.0,10,162.0,42.0,20.0,159,176.0,379.0,184.0,70.0,6.0,16.0,187.0,197,van
1,91,41.0,84.0,141.0,57.0,9,149.0,45.0,19.0,143,170.0,330.0,158.0,72.0,9.0,14.0,189.0,199,van
2,104,50.0,106.0,209.0,66.0,10,207.0,32.0,23.0,158,223.0,635.0,220.0,73.0,14.0,9.0,188.0,196,car
3,93,41.0,82.0,159.0,63.0,9,144.0,46.0,19.0,143,160.0,309.0,127.0,63.0,6.0,10.0,199.0,207,van
4,85,44.0,70.0,205.0,103.0,52,149.0,45.0,19.0,144,241.0,325.0,188.0,127.0,9.0,11.0,180.0,183,bus


# Exploration and Preprocessing Data

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 846 entries, 0 to 845
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   compactness                  846 non-null    int64  
 1   circularity                  841 non-null    float64
 2   distance_circularity         842 non-null    float64
 3   radius_ratio                 840 non-null    float64
 4   pr.axis_aspect_ratio         844 non-null    float64
 5   max.length_aspect_ratio      846 non-null    int64  
 6   scatter_ratio                845 non-null    float64
 7   elongatedness                845 non-null    float64
 8   pr.axis_rectangularity       843 non-null    float64
 9   max.length_rectangularity    846 non-null    int64  
 10  scaled_variance              843 non-null    float64
 11  scaled_variance.1            844 non-null    float64
 12  scaled_radius_of_gyration    844 non-null    float64
 13  scaled_radius_of_gyr

In [ ]:
df.shape

(846, 19)

In [ ]:
df.isna().sum()

,0
compactness,0
circularity,5
distance_circularity,4
radius_ratio,6
pr.axis_aspect_ratio,2
max.length_aspect_ratio,0
scatter_ratio,1
elongatedness,1
pr.axis_rectangularity,3
max.length_rectangularity,0


In [ ]:
# Remove missing value (Null value)

df = df.dropna()

In [ ]:
df.head()

,compactness,circularity,distance_circularity,radius_ratio,pr.axis_aspect_ratio,max.length_aspect_ratio,scatter_ratio,elongatedness,pr.axis_rectangularity,max.length_rectangularity,scaled_variance,scaled_variance.1,scaled_radius_of_gyration,scaled_radius_of_gyration.1,skewness_about,skewness_about.1,skewness_about.2,hollows_ratio,class
0,95,48.0,83.0,178.0,72.0,10,162.0,42.0,20.0,159,176.0,379.0,184.0,70.0,6.0,16.0,187.0,197,van
1,91,41.0,84.0,141.0,57.0,9,149.0,45.0,19.0,143,170.0,330.0,158.0,72.0,9.0,14.0,189.0,199,van
2,104,50.0,106.0,209.0,66.0,10,207.0,32.0,23.0,158,223.0,635.0,220.0,73.0,14.0,9.0,188.0,196,car
3,93,41.0,82.0,159.0,63.0,9,144.0,46.0,19.0,143,160.0,309.0,127.0,63.0,6.0,10.0,199.0,207,van
4,85,44.0,70.0,205.0,103.0,52,149.0,45.0,19.0,144,241.0,325.0,188.0,127.0,9.0,11.0,180.0,183,bus


In [ ]:
df.groupby('class').size()

,0
class,
bus,205
car,413
van,195


# Classification Models

In [ ]:
df.columns

Index(['compactness', 'circularity', 'distance_circularity', 'radius_ratio',
       'pr.axis_aspect_ratio', 'max.length_aspect_ratio', 'scatter_ratio',
       'elongatedness', 'pr.axis_rectangularity', 'max.length_rectangularity',
       'scaled_variance', 'scaled_variance.1', 'scaled_radius_of_gyration',
       'scaled_radius_of_gyration.1', 'skewness_about', 'skewness_about.1',
       'skewness_about.2', 'hollows_ratio', 'class'],
      dtype='object')

In [ ]:
# Feature & Target Selection
Features = ['compactness', 'circularity', 'distance_circularity', 'radius_ratio', 'pr.axis_aspect_ratio', 'max.length_aspect_ratio', 'scatter_ratio', 'elongatedness',
            'pr.axis_rectangularity', 'max.length_rectangularity', 'scaled_variance', 'scaled_variance.1', 'scaled_radius_of_gyration', 'scaled_radius_of_gyration.1',
            'skewness_about', 'skewness_about.1', 'skewness_about.2', 'hollows_ratio']

Target = ['class']

# Features & Target Selection
vehicle_X = df[Features]
vehicle_y = df[Target]

In [ ]:
# Feature Scaling
scaler = StandardScaler()
scaler_vehicle_X = pd.DataFrame(scaler.fit_transform(vehicle_X))
scaler_vehicle_X.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,0.163231,0.520408,0.060669,0.264970,1.283254,0.299721,-0.198517,0.129648,-0.217151,0.766312,-0.397397,-0.339014,0.301676,-0.321192,-0.071523,0.371287,-0.321809,0.171837
1,-0.322874,-0.619123,0.124067,-0.836393,-0.599253,0.085785,-0.591720,0.514333,-0.606014,-0.337462,-0.590034,-0.618754,-0.502972,-0.053505,0.538425,0.147109,0.003400,0.442318
2,1.256966,0.845988,1.518823,1.187734,0.530251,0.299721,1.162569,-1.152637,0.949438,0.697326,1.111591,1.122486,1.415804,0.080339,1.555006,-0.413338,-0.159204,0.036596
3,-0.079822,-0.619123,-0.002729,-0.300595,0.153750,0.085785,-0.742952,0.642562,-0.606014,-0.337462,-0.911095,-0.738643,-1.462359,-1.258099,-0.071523,-0.301249,1.629444,1.524243
4,-1.052030,-0.130753,-0.763506,1.068668,5.173770,9.285029,-0.591720,0.514333,-0.606014,-0.268476,1.689501,-0.647299,0.425468,7.307905,0.538425,-0.189159,-1.460039,-1.721531


In [ ]:
# Dataset Split [Train = 70%, test = 30%]
X_train, X_test, y_train, y_test = train_test_split(
    scaler_vehicle_X, vehicle_y, stratify=vehicle_y, test_size=.3, random_state=3)


In [ ]:
ran = RandomForestClassifier()

In [ ]:
model = ran.fit(X_train, y_train)
model

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier()

In [ ]:
y_pred = model.predict(X_test)
y_true = y_test

In [ ]:
y_pred

array(['van', 'car', 'van', 'bus', 'car', 'van', 'car', 'car', 'car',
       'car', 'bus', 'car', 'car', 'car', 'car', 'van', 'car', 'van',
       'car', 'car', 'van', 'van', 'car', 'car', 'bus', 'car', 'car',
       'car', 'van', 'van', 'car', 'car', 'car', 'bus', 'bus', 'car',
       'van', 'bus', 'bus', 'van', 'car', 'car', 'van', 'bus', 'car',
       'car', 'car', 'car', 'car', 'car', 'van', 'car', 'van', 'car',
       'car', 'bus', 'bus', 'van', 'car', 'car', 'bus', 'bus', 'bus',
       'bus', 'car', 'bus', 'car', 'bus', 'car', 'car', 'bus', 'van',
       'car', 'car', 'van', 'bus', 'car', 'bus', 'van', 'van', 'bus',
       'bus', 'car', 'van', 'bus', 'car', 'bus', 'car', 'van', 'van',
       'car', 'van', 'van', 'van', 'car', 'car', 'car', 'bus', 'car',
       'car', 'car', 'van', 'bus', 'bus', 'van', 'bus', 'van', 'car',
       'van', 'bus', 'car', 'van', 'car', 'car', 'van', 'car', 'car',
       'car', 'bus', 'bus', 'van', 'van', 'car', 'car', 'bus', 'car',
       'bus', 'car',

In [ ]:
y_test

,class
7,van
386,car
128,van
564,bus
361,car
...,...
515,car
780,car
23,bus
784,van


In [ ]:
def get_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall= recall_score(y_true, y_pred, average='weighted')
    f1= f1_score(y_true, y_pred, average='weighted')
    return { 'accuracy': accuracy , "precision": precision, "recall": recall, 'f1_score': f1}

In [ ]:
single_model_metrics = get_metrics(y_true, y_pred)

single_model_metrics

{'accuracy': 0.9467213114754098,
 'precision': 0.9467547081445439,
 'recall': 0.9467213114754098,
 'f1_score': 0.9466743621392023}

# Bagging

In [ ]:
def withBagging(X,y, num_bags=5):
    # Ensure y is a 1D array for train_test_split and subsequent operations
    y_flat = y.values.ravel() if isinstance(y, pd.DataFrame) else y

    # Unpack the tuple containing the split data
    # X_train and X_test will be DataFrames, y_train and y_test will be 1D numpy arrays
    X_train, X_test, y_train, y_test = train_test_split(X, y_flat, stratify=y_flat, test_size=.3, random_state=3)

    # Reset indices of X_train and convert y_train to a Series before combining
    X_train_reset = X_train.reset_index(drop=True)
    y_train_series = pd.Series(y_train, name='Class') # Name the series 'Class'

    # Combine X_train and y_train into a single DataFrame for bootstrap sampling
    # Ensure indices are aligned by concatenating after reset_index
    train = pd.concat([X_train_reset, y_train_series], axis=1)

    allResult = []
    for bag in range(num_bags):
        # Create a Bootstrap Sample (sampling with replacement)
        sample = train.sample(frac=1, replace=True, random_state=bag) # Added random_state for reproducibility

        # Split sample back into features and target
        Xtrain_bag = sample.drop('Class', axis=1)
        ytrain_bag = sample['Class'] # Access by column name

        # Initialize and fit the classifier for this bag
        ran = RandomForestClassifier(random_state=bag) # Added random_state for reproducibility
        ran.fit(Xtrain_bag, ytrain_bag)

        # Predict and evaluate
        y_pred = ran.predict(X_test)
        y_true = y_test # y_test is already a 1D numpy array

        # Append the metrics for this specific bag
        allResult.append(get_metrics(y_true, y_pred))

    # Return the average of all bags as a dictionary rounded to 3 decimal places
    return pd.DataFrame(allResult).mean().round(3).to_dict()

In [ ]:
withBagging(scaler_vehicle_X, vehicle_y, num_bags=5)

{'accuracy': 0.939, 'precision': 0.939, 'recall': 0.939, 'f1_score': 0.938}

In [ ]:
for num_bags in [5, 10, 15, 20, 50]:

    withBaggingResults = withBagging(scaler_vehicle_X, vehicle_y, num_bags=num_bags)

    print(f'results for {num_bags} bags ...')
    print(f'{single_model_metrics=}')
    print(f'{withBaggingResults=}')
    print('\n')

results for 5 bags ...
single_model_metrics={'accuracy': 0.9467213114754098, 'precision': 0.9467547081445439, 'recall': 0.9467213114754098, 'f1_score': 0.9466743621392023}
withBaggingResults={'accuracy': 0.939, 'precision': 0.939, 'recall': 0.939, 'f1_score': 0.938}


results for 10 bags ...
single_model_metrics={'accuracy': 0.9467213114754098, 'precision': 0.9467547081445439, 'recall': 0.9467213114754098, 'f1_score': 0.9466743621392023}
withBaggingResults={'accuracy': 0.934, 'precision': 0.934, 'recall': 0.934, 'f1_score': 0.934}


results for 15 bags ...
single_model_metrics={'accuracy': 0.9467213114754098, 'precision': 0.9467547081445439, 'recall': 0.9467213114754098, 'f1_score': 0.9466743621392023}
withBaggingResults={'accuracy': 0.937, 'precision': 0.937, 'recall': 0.937, 'f1_score': 0.936}


results for 20 bags ...
single_model_metrics={'accuracy': 0.9467213114754098, 'precision': 0.9467547081445439, 'recall': 0.9467213114754098, 'f1_score': 0.9466743621392023}
withBaggingResults

# Stacking

In [ ]:
def getTrainTestData(X: np.ndarray, y: Series) -> tuple:
    return train_test_split(X, y, stratify=y, test_size=.3, random_state=3)

In [ ]:
vehicle_training_datas = getTrainTestData(scaler_vehicle_X, vehicle_y)

In [ ]:
X_train, X_test, y_train, y_test= vehicle_training_datas
X_test

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
6,-0.444400,-0.293543,-1.017098,-0.360128,0.404751,0.085785,-0.954676,0.899019,-0.994877,-0.130505,-0.846883,-0.898494,-0.317284,-0.722724,-0.681472,-1.085874,0.653818,0.848040
358,0.649335,-0.293543,0.441057,0.771002,0.153750,-0.128151,0.527395,-0.767951,0.560575,-0.268476,0.694211,0.471662,-0.255388,-0.588880,-0.071523,0.035019,0.328609,0.442318
118,0.406283,0.032037,0.377659,0.116137,0.655752,0.299721,-0.349749,0.257876,-0.217151,0.628340,-0.493716,-0.418940,-0.533920,-0.722724,0.335109,-0.077070,0.491213,0.577559
531,0.527809,-1.595864,-0.763506,0.384036,0.781252,-0.769959,-0.410242,0.001420,-0.606014,-1.648193,-0.140548,-0.384686,-1.771839,-0.187348,-0.681472,1.043823,2.117257,1.253762
334,0.892387,0.194828,1.455425,0.771002,-0.097251,0.299721,0.890352,-0.896180,0.949438,0.145439,0.790530,0.797074,0.642104,-0.856567,0.131793,2.837252,0.491213,1.253762
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,-2.145765,-1.107494,-1.524282,-1.312659,-0.473753,-0.769959,-1.075662,1.283704,-0.994877,-1.441236,-1.167944,-1.024092,-0.905295,1.954153,-1.291420,0.931734,-1.947853,-1.586291
747,1.864596,0.357618,0.884843,1.098434,0.279250,0.085785,0.890352,-1.024408,0.949438,0.145439,0.983166,0.842746,0.518312,-0.321192,-0.478156,0.035019,0.166004,0.307078
20,-0.808978,-0.944703,-0.763506,-0.628027,-0.097251,-0.342087,-0.773198,0.642562,-0.994877,-0.820363,-0.782671,-0.750061,-1.029087,-0.455036,-1.088104,-1.197963,0.491213,0.442318
751,-0.808978,-1.270284,-1.904670,-1.580558,-0.975755,-0.556023,-1.620096,2.181304,-1.383740,-1.510222,-1.713749,-1.383758,-0.936243,0.214183,0.945058,1.043823,-0.484413,-0.774847


In [ ]:
decisiontree_clf = DecisionTreeClassifier()
svm_clf          = SVC(probability=True)
logisticreg_clf  = LogisticRegression()
randomforest_clf = RandomForestClassifier()
linear_clf       = LinearDiscriminantAnalysis()
knn_clf   = KNeighborsClassifier()
naivebayes_clf   = GaussianNB()

In [ ]:
classifiers = [randomforest_clf, linear_clf, knn_clf, naivebayes_clf]
classifiers

[RandomForestClassifier(),
 LinearDiscriminantAnalysis(),
 KNeighborsClassifier(),
 GaussianNB()]

In [ ]:
def get_base_predictions(X, y, classifiers):
    base_prediction = []
    X_train, X_test, y_train, y_test = train_test_split(X, y,stratify=y, test_size=.3, random_state=3)
    for classifier in classifiers:
        classifier.fit(X_train, y_train)
        base_prediction.append(DataFrame(classifier.predict_proba(X_test)))
    return  pd.concat(base_prediction, axis=1)

In [ ]:
base_predictions = get_base_predictions(vehicle_X, vehicle_y, classifiers)
base_predictions

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for exa

,0,1,2,0,1,2,0,1,2,0,1,2
0,0.01,0.00,0.99,0.004332,0.005259,0.990410,0.0,0.0,1.0,0.002197,0.000180,9.976229e-01
1,0.05,0.95,0.00,0.000065,0.999932,0.000004,0.2,0.8,0.0,0.053084,0.946916,1.321637e-11
2,0.02,0.06,0.92,0.000010,0.000028,0.999961,0.0,0.0,1.0,0.090415,0.708358,2.012275e-01
3,0.89,0.10,0.01,0.999991,0.000004,0.000005,0.6,0.4,0.0,0.521716,0.252285,2.259986e-01
4,0.00,1.00,0.00,0.000001,0.999987,0.000012,0.0,1.0,0.0,0.000001,0.999999,2.186887e-22
...,...,...,...,...,...,...,...,...,...,...,...,...
239,0.00,0.99,0.01,0.193918,0.805560,0.000522,0.0,1.0,0.0,0.499543,0.000004,5.004528e-01
240,0.00,1.00,0.00,0.000128,0.999846,0.000025,0.2,0.8,0.0,0.003682,0.996318,1.643114e-22
241,0.20,0.60,0.20,0.421467,0.036584,0.541949,0.4,0.2,0.4,0.005536,0.000313,9.941513e-01
242,0.00,0.32,0.68,0.000351,0.476037,0.523612,0.0,0.0,1.0,0.000448,0.000070,9.994828e-01


In [ ]:
def train_test_num(number: int):
    whole, frac = divmod(number, 3)
    training_num = 2 * whole
    testing_num = whole + frac
    return training_num, testing_num

In [ ]:
training_num, testing_num = train_test_num(y_test.shape[0])
training_num, testing_num

(162, 82)

In [ ]:
X_train_meta = base_predictions.head(training_num)
X_test_meta  = base_predictions.tail(testing_num)

In [ ]:
randomforest_clf.fit(X_train_meta, y_test.head(training_num))

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier()

In [ ]:
def get_metrics(y_true, y_pred):

    precision = precision_score(y_true, y_pred, average='weighted')
    recall= recall_score(y_true, y_pred, average='weighted')
    f1= f1_score(y_true, y_pred, average='weighted')
    accuracy = accuracy_score(y_true, y_pred)

    return {"precision": precision, "recall": recall, 'f1_score': f1, 'accuracy': accuracy }

In [ ]:
get_metrics(randomforest_clf.predict(X_test_meta), np.array(y_test.tail(testing_num)))

{'precision': 0.9883351007423118,
 'recall': 0.9878048780487805,
 'f1_score': 0.9878748142320133,
 'accuracy': 0.9878048780487805}

Metrics for all Models

In [ ]:
X_train, X_test, y_train, y_test = vehicle_training_datas

# Convert y_train and y_test to 1D arrays to avoid DataConversionWarning
y_train_flat = y_train.values.ravel()
y_test_flat = y_test.values.ravel()

for classifier in classifiers:
    print(f"\nEvaluating {classifier.__class__.__name__}:")
    classifier.fit(X_train, y_train_flat)
    y_pred = classifier.predict(X_test)
    metrics = get_metrics(y_test_flat, y_pred)
    print(metrics)


Evaluating RandomForestClassifier:
{'precision': 0.9589646246764452, 'recall': 0.9590163934426229, 'f1_score': 0.9589657274858522, 'accuracy': 0.9590163934426229}

Evaluating LinearDiscriminantAnalysis:
{'precision': 0.9265276471378662, 'recall': 0.9262295081967213, 'f1_score': 0.926288214872781, 'accuracy': 0.9262295081967213}

Evaluating KNeighborsClassifier:
{'precision': 0.921816152402865, 'recall': 0.9221311475409836, 'f1_score': 0.9219429793300071, 'accuracy': 0.9221311475409836}

Evaluating GaussianNB:
{'precision': 0.7204127121081393, 'recall': 0.6434426229508197, 'f1_score': 0.6418156494005266, 'accuracy': 0.6434426229508197}
